# 02. 전처리 — 필터링 / 기준값 / X·Y 구간 분리 / Y값 계산
- 01_data_collection.ipynb 완료 후 실행
- 순서: 선발투수 필터링 → 직전 시즌 기준값 → X/Y 구간 분리 → Y값 계산

In [ ]:
# ── 환경 감지 ──────────────────────────────────────────────
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR   = '/content/drive/MyDrive/MLB_pitcher/data'
else:
    BASE_DIR   = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

RAW_DIR      = os.path.join(BASE_DIR, 'raw')
INTERIM_DIR  = os.path.join(BASE_DIR, 'interim')
os.makedirs(INTERIM_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'RAW   : {RAW_DIR}')
print(f'INTERIM: {INTERIM_DIR}')

In [ ]:
import pandas as pd
import numpy as np

SEASONS = [2021, 2022, 2023, 2024, 2025]

## 1. 전체 데이터 로드

In [ ]:
dfs = []
for yr in SEASONS:
    path = os.path.join(RAW_DIR, f'statcast_{yr}.parquet')
    if not os.path.exists(path):
        print(f'[{yr}] 파일 없음 — 건너뜀')
        continue
    df = pd.read_parquet(path)
    df['season'] = yr
    dfs.append(df)
    print(f'[{yr}] {len(df):,}행 로드')

raw = pd.concat(dfs, ignore_index=True)
print(f'\n전체: {len(raw):,}행')

## 2. 선발투수 필터링

In [ ]:
# ── 2-1. 정규시즌만 ────────────────────────────────────────
df = raw[raw['game_type'] == 'R'].copy()
print(f'정규시즌 필터 후: {len(df):,}행')

In [ ]:
# ── 2-2. 선발투수 판별 ─────────────────────────────────────
# 1이닝부터 등판 + 일정 이닝(MIN_INNINGS) 이상 소화한 투수만
MIN_INNINGS = 3  # 조정 가능

# 경기별 투수 정보 집계
game_pitcher = (
    df.groupby(['game_pk', 'pitcher'])
    .agg(
        min_inning  = ('inning', 'min'),
        max_inning  = ('inning', 'max'),
        total_pitch = ('pitch_number', 'count'),
    )
    .reset_index()
)

# 선발투수 조건: 1이닝부터 + MIN_INNINGS 이상
starters = game_pitcher[
    (game_pitcher['min_inning'] == 1) &
    (game_pitcher['max_inning'] >= MIN_INNINGS)
][['game_pk', 'pitcher']].copy()
starters['is_starter'] = True

df = df.merge(starters, on=['game_pk', 'pitcher'], how='inner')
print(f'선발투수 필터 후: {len(df):,}행')
print(f'고유 (경기, 투수) 수: {df.groupby(["game_pk","pitcher"]).ngroups:,}')

In [ ]:
# ── 2-3. 구종 없는 행 제거 ────────────────────────────────
df = df[df['pitch_type'].notna() & (df['pitch_type'] != '')].copy()
print(f'구종 결측 제거 후: {len(df):,}행')

# 저장
out = os.path.join(INTERIM_DIR, 'starters_all.parquet')
df.to_parquet(out, index=False)
print(f'저장 완료 → {out}')

## 3. 직전 시즌 기준값 계산 (Lookup Table)

In [ ]:
# ── 구종 그룹화 ────────────────────────────────────────────
PITCH_GROUP = {
    'FF': 'Fastball', 'SI': 'Fastball', 'FC': 'Fastball',
    'SL': 'Breaking', 'CU': 'Breaking', 'KC': 'Breaking',
    'CH': 'Offspeed', 'FS': 'Offspeed',
}

df['pitch_group'] = df['pitch_type'].map(PITCH_GROUP)

# 그룹 매핑 안 된 구종 확인
unmapped = df[df['pitch_group'].isna()]['pitch_type'].value_counts()
print('매핑 안 된 구종:')
print(unmapped.head(10))

In [ ]:
# ── 투수별 / 구종별 / 시즌별 기준값 계산 ─────────────────
baseline = (
    df.groupby(['pitcher', 'pitch_type', 'season'])
    .agg(
        avg_speed    = ('release_speed',     'mean'),
        avg_spin     = ('release_spin_rate', 'mean'),
        avg_ext      = ('release_extension', 'mean'),
        avg_pos_x    = ('release_pos_x',     'mean'),
        avg_pos_z    = ('release_pos_z',     'mean'),
        pitch_count  = ('pitch_type',        'count'),
    )
    .reset_index()
)

# 투구 수 20개 미만 구종은 기준값 신뢰도 낮으므로 제외
baseline = baseline[baseline['pitch_count'] >= 20]

print(f'기준값 테이블: {len(baseline):,}행')
baseline.head()

In [ ]:
# ── 직전 시즌 기준값으로 변환 ─────────────────────────────
# delta feature용: 올해 시즌 기준값 → 직전 시즌 기준값으로 shift
baseline['prev_season']      = baseline['season'] + 1  # 다음 시즌에서 이 값을 참조

lookup = baseline.rename(columns={
    'prev_season': 'season',
    'avg_speed':   'prev_avg_speed',
    'avg_spin':    'prev_avg_spin',
    'avg_ext':     'prev_avg_ext',
    'avg_pos_x':   'prev_avg_pos_x',
    'avg_pos_z':   'prev_avg_pos_z',
})[['pitcher', 'pitch_type', 'season',
    'prev_avg_speed', 'prev_avg_spin', 'prev_avg_ext',
    'prev_avg_pos_x', 'prev_avg_pos_z']]

out = os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet')
lookup.to_parquet(out, index=False)
print(f'lookup table 저장 완료 → {out}')
lookup.head()

## 4. X / Y 구간 분리 함수

In [ ]:
def split_xy(game_df, mode='batter', n=9):
    """
    한 경기 한 투수의 투구 데이터를 X구간 / Y구간으로 분리

    Parameters
    ----------
    game_df : DataFrame  — 단일 (game_pk, pitcher) 그룹
    mode    : 'pitch'    — 초반 n구
              'inning'   — 초반 n이닝
              'batter'   — 초반 n타자
    n       : 구간 기준값

    Returns
    -------
    x_df, y_df
    """
    g = game_df.sort_values(['at_bat_number', 'pitch_number'])

    if mode == 'pitch':
        # 투구 순서 번호 부여
        g = g.reset_index(drop=True)
        mask = g.index < n

    elif mode == 'inning':
        mask = g['inning'] <= n

    elif mode == 'batter':
        mask = g['at_bat_number'] <= g['at_bat_number'].min() + n - 1

    else:
        raise ValueError(f'mode는 pitch / inning / batter 중 하나여야 합니다: {mode}')

    x_df = g[mask].copy()
    y_df = g[~mask].copy()
    return x_df, y_df


print('split_xy 함수 정의 완료')
print('사용 예시:')
print('  x, y = split_xy(game_df, mode="batter", n=9)   # 초반 9타자')
print('  x, y = split_xy(game_df, mode="inning", n=2)   # 초반 2이닝')
print('  x, y = split_xy(game_df, mode="pitch",  n=15)  # 초반 15구')

## 5. Y값 계산 (wOBA against)

In [ ]:
def calc_y(y_df):
    """
    Y구간 투구 데이터로 wOBA against 계산

    - woba_value가 있는 행 = 타석 종료 행
    - 삼진(strikeout): woba_value = 0
    - 볼넷(walk): woba_value = 0.69  (Statcast 기본값 그대로)
    - 인플레이: woba_value 그대로
    - woba_denom = 0인 타석(고의사구 등)은 분모에서 제외

    Returns
    -------
    float : 경기 단위 wOBA against (Y구간 없으면 NaN)
    """
    # 타석 종료 행만 추출 (woba_denom이 1인 행)
    ab_end = y_df[y_df['woba_denom'] == 1].copy()

    if len(ab_end) == 0:
        return np.nan

    # 삼진은 woba_value가 NaN으로 들어오는 경우가 있으므로 0으로 채움
    # (events == 'strikeout')
    ab_end['woba_value'] = ab_end['woba_value'].fillna(0)

    woba = ab_end['woba_value'].sum() / ab_end['woba_denom'].sum()
    return round(woba, 4)


print('calc_y 함수 정의 완료')

In [ ]:
# ── 전체 데이터에 적용 (mode / n 조합 실험) ───────────────
# 우선 batter / n=9 로 테스트
MODE = 'batter'
N    = 9

records = []
groups  = df.groupby(['game_pk', 'pitcher', 'season'])

for (game_pk, pitcher, season), game_df in groups:
    x_df, y_df = split_xy(game_df, mode=MODE, n=N)

    # Y구간이 너무 짧으면 skip (5타석 미만)
    if y_df['woba_denom'].eq(1).sum() < 5:
        continue

    y_val = calc_y(y_df)

    records.append({
        'game_pk':  game_pk,
        'pitcher':  pitcher,
        'season':   season,
        'x_pitches': len(x_df),
        'y_ab':     y_df['woba_denom'].eq(1).sum(),
        'y_woba':   y_val,
        'mode':     MODE,
        'n':        N,
    })

result = pd.DataFrame(records)
print(f'결과: {len(result):,}경기')
print(f'y_woba 분포:')
print(result['y_woba'].describe())

In [ ]:
# ── 저장 ──────────────────────────────────────────────────
out = os.path.join(INTERIM_DIR, f'y_woba_{MODE}{N}.parquet')
result.to_parquet(out, index=False)
print(f'저장 완료 → {out}')
result.head()